# GPTQ attention-only notebook — Perplexity + GSM8K + ARC-Challenge

This notebook runs the same execution order as the full-quant notebook, but for **Qwen2-1.5B attention-only GPTQ quantization**:

1. quantize and save the model artifact
2. evaluate **Perplexity** on WikiText-2
3. evaluate **GSM8K**
4. evaluate **ARC-Challenge**

It saves:
- quantized GPTQ artifact folder
- quantized model zip
- perplexity / GSM8K / ARC-Challenge metrics and predictions
- summary table row
- results-only zip


In [ ]:

# Colab / notebook setup
!pip install -U pip setuptools wheel
!pip install -U gptqmodel
!pip install -q torch transformers accelerate datasets pandas pyarrow safetensors tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 87.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 72.1 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.4/668.4 kB 29.3 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 188.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 162.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 224.3 MB/s  0:00:00
   ━━━━━━━━━━━━━

In [ ]:

# Restart once after installation in a fresh runtime, then skip this cell on reruns.
import IPython
IPython.Application.instance().kernel.do_shutdown(True)


{'status': 'ok', 'restart': True}

In [ ]:

from __future__ import annotations

import json
import logging
import math
import os
import re
import time
import zipfile
from pathlib import Path
from typing import Any

import pandas as pd
import torch
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from gptqmodel import GPTQModel, QuantizeConfig

# ------------------------------
# Logging
# ------------------------------
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-7s | %(message)s',
    force=True,
)
logger = logging.getLogger('gptq_attn_only_quant')

PROJECT_ROOT = Path.cwd().resolve()
VARIANT = 'attn_only_quant'
RUN_NAME = 'gptq_w4_g128_attn_only_quant'
BASE_MODEL = 'Qwen/Qwen2-1.5B'

cfg = {
    'base_model': BASE_MODEL,
    'calibration': {
        'dataset': 'wikitext',
        'dataset_name': 'wikitext-2-raw-v1',
        'split': 'train',
        'num_samples': 512,
        'max_length': 2048,
        'seed': 42,
    },
    'quant': {
        'bits': 4,
        'group_size': 128,
        'desc_act': False,
    },
    'eval': {
        'perplexity': {
            'dataset': 'wikitext',
            'dataset_name': 'wikitext-2-raw-v1',
            'split': 'test',
            'max_length': 2048,
            'stride': 1024,
            'max_eval_tokens': 131072,
            'log_every': 25,
        },
        'gsm8k': {
            'num_fewshot': 8,
            'num_samples': 300,
            'max_new_tokens': 256,
            'log_every': 25,
        },
        'arc_challenge': {
            'num_samples': 500,
            'max_new_tokens': 5,
            'log_every': 50,
        },
    },
    'paths': {
        'artifacts_dir': str(PROJECT_ROOT / 'artifacts'),
        'results_dir': str(PROJECT_ROOT / 'results' / RUN_NAME),
        'calibration_dir': str(PROJECT_ROOT / 'calibration_data'),
        'bundles_dir': str(PROJECT_ROOT / 'zip_bundles' / RUN_NAME),
    },
}

PATHS = {k: Path(v) for k, v in cfg['paths'].items()}
for p in PATHS.values():
    p.mkdir(parents=True, exist_ok=True)

ARTIFACT_DIR = PATHS['artifacts_dir'] / 'gptq_w4_g128' / VARIANT
RESULTS_DIR = PATHS['results_dir'] / VARIANT
BUNDLE_DIR = PATHS['bundles_dir'] / VARIANT
CALIB_DIR = PATHS['calibration_dir']
for p in [ARTIFACT_DIR, RESULTS_DIR, BUNDLE_DIR, CALIB_DIR]:
    p.mkdir(parents=True, exist_ok=True)


def build_dynamic_config(variant: str) -> dict | None:
    if variant == 'full_quant':
        return None
    if variant == 'attn_only_quant':
        return {r'-:.*mlp\.(gate|up|down)_proj.*': {}}
    if variant == 'mlp_only_quant':
        return {r'-:.*self_attn\.(q|k|v|o)_proj.*': {}}
    raise ValueError(f'Unknown variant: {variant}')

print('Working directory:', PROJECT_ROOT)
print('Artifact dir:', ARTIFACT_DIR)
print('Results dir:', RESULTS_DIR)
print('Bundle dir:', BUNDLE_DIR)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

# ------------------------------
# Small utilities
# ------------------------------
def save_json(obj: Any, path: str | Path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w') as f:
        json.dump(obj, f, indent=2)
    return path


def read_json(path: str | Path, default=None):
    path = Path(path)
    if not path.exists():
        return default
    with open(path) as f:
        return json.load(f)


def zip_directory(source_dir: str | Path, zip_path: str | Path, *, exclude_suffixes=('.lock',), exclude_dirs=('.ipynb_checkpoints',)):
    source_dir = Path(source_dir).resolve()
    zip_path = Path(zip_path).resolve()
    zip_path.parent.mkdir(parents=True, exist_ok=True)

    tmp_zip = zip_path.with_suffix(zip_path.suffix + '.tmp')
    if tmp_zip.exists():
        tmp_zip.unlink()
    if zip_path.exists():
        zip_path.unlink()

    with zipfile.ZipFile(tmp_zip, 'w', compression=zipfile.ZIP_DEFLATED, allowZip64=True) as zf:
        for path in sorted(source_dir.rglob('*')):
            if path.is_dir():
                continue
            if any(part in exclude_dirs for part in path.parts):
                continue
            if path.resolve() in {zip_path, tmp_zip}:
                continue
            if path.suffix in exclude_suffixes:
                continue
            zf.write(path, arcname=str(path.relative_to(source_dir)))

    tmp_zip.replace(zip_path)
    size_mb = zip_path.stat().st_size / (1024 ** 2)
    logger.info('Created zip: %s (%.2f MB)', zip_path, size_mb)
    return zip_path


def find_gptq_model_basename(model_dir: str | Path) -> str | None:
    model_dir = Path(model_dir)
    candidates = sorted(model_dir.glob('*.safetensors'))
    if not candidates:
        return None
    preferred = [p for p in candidates if 'gptq' in p.name.lower()]
    return (preferred[0] if preferred else candidates[0]).name


def estimate_num_params_from_config(model_dir: str | Path) -> int | None:
    cfg_path = Path(model_dir) / 'config.json'
    if not cfg_path.exists():
        return None
    cfg_json = json.loads(cfg_path.read_text())
    hidden_size = cfg_json.get('hidden_size')
    num_hidden_layers = cfg_json.get('num_hidden_layers')
    intermediate_size = cfg_json.get('intermediate_size')
    vocab_size = cfg_json.get('vocab_size')
    if not all(v is not None for v in [hidden_size, num_hidden_layers, intermediate_size, vocab_size]):
        return None
    rough_params = (
        vocab_size * hidden_size +
        num_hidden_layers * (4 * hidden_size * hidden_size) +
        num_hidden_layers * (3 * hidden_size * intermediate_size)
    )
    return int(rough_params)


def model_size_gb(model_dir: str | Path) -> float:
    files = list(Path(model_dir).glob('*.safetensors'))
    total_bytes = sum(p.stat().st_size for p in files)
    return total_bytes / (1024 ** 3)


def bits_per_param(model_dir: str | Path) -> float | None:
    files = list(Path(model_dir).glob('*.safetensors'))
    if not files:
        return None
    total_bytes = sum(p.stat().st_size for p in files)
    n_params = estimate_num_params_from_config(model_dir)
    if not n_params:
        return None
    return 8.0 * total_bytes / n_params


def get_results_state() -> dict:
    state = read_json(RESULTS_DIR / 'state.json', default={})
    return state or {}


def save_results_state(state: dict):
    save_json(state, RESULTS_DIR / 'state.json')
    return state


def update_summary_table(state: dict):
    size_gb = model_size_gb(ARTIFACT_DIR) if ARTIFACT_DIR.exists() else None
    bpp = bits_per_param(ARTIFACT_DIR) if ARTIFACT_DIR.exists() else None
    peak_candidates = []
    for key in ['perplexity', 'gsm8k', 'arc_challenge']:
        if key in state and isinstance(state[key], dict):
            val = state[key].get('peak_vram_gb')
            if val is not None:
                peak_candidates.append(val)

    summary_row = {
        'variant': VARIANT,
        'compression': f"GPTQ W{cfg['quant']['bits']} g{cfg['quant']['group_size']}",
        'model_size_gb': round(size_gb, 4) if size_gb is not None else None,
        'peak_vram_gb': round(max(peak_candidates), 4) if peak_candidates else None,
        'bits_per_param': round(bpp, 4) if bpp is not None else None,
        'perplexity': state.get('perplexity', {}).get('perplexity'),
        'gsm8k_accuracy': state.get('gsm8k', {}).get('accuracy'),
        'arc_accuracy': state.get('arc_challenge', {}).get('accuracy'),
    }
    save_json(summary_row, RESULTS_DIR / 'summary.json')
    pd.DataFrame([summary_row]).to_csv(RESULTS_DIR / 'summary.csv', index=False)
    return summary_row


def _is_gptq_checkpoint(model_path: str | Path) -> bool:
    cfg_path = Path(model_path) / 'config.json'
    if not cfg_path.exists():
        return False
    cfg_json = json.loads(cfg_path.read_text())
    return cfg_json.get('quantization_config', {}).get('quant_method') == 'gptq'


def load_quantized_model_and_tokenizer(model_path: str | Path):
    model_path = Path(model_path)
    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = 'left'

    if not _is_gptq_checkpoint(model_path):
        raise ValueError(f'{model_path} is not a GPTQ checkpoint.')

    model_basename = find_gptq_model_basename(model_path)
    if model_basename is None:
        raise FileNotFoundError(f'No GPTQ safetensors found in {model_path}')

    model = GPTQModel.load(
        str(model_path),
        device_map='auto',
        trust_remote_code=True,
        model_basename=model_basename,
    )
    model.eval()
    return model, tokenizer


def get_model_device(model):
    if hasattr(model, 'device'):
        return model.device
    if hasattr(model, 'hf_device_map') and model.hf_device_map:
        first_device = next(iter(model.hf_device_map.values()))
        return torch.device(first_device)
    return torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')


def generate_text(model, tokenizer, prompt: str, *, max_new_tokens: int):
    device = get_model_device(model)
    encoded = tokenizer(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=min(getattr(tokenizer, 'model_max_length', 4096), 4096),
        padding=False,
    )
    input_ids = encoded['input_ids'].to(device)
    attention_mask = encoded['attention_mask'].to(device)

    with torch.inference_mode():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )
    gen_ids = outputs[0][input_ids.shape[-1]:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True)


ANSWER_RE = re.compile(r'####\s*(-?[\d,]+\.?\d*)')


def extract_numeric_answer(text: str):
    m = ANSWER_RE.search(text)
    if m:
        try:
            return float(m.group(1).replace(',', ''))
        except ValueError:
            return None
    nums = re.findall(r'-?[\d,]+\.?\d*', text)
    if not nums:
        return None
    try:
        return float(nums[-1].replace(',', ''))
    except ValueError:
        return None


def format_seconds(sec: float) -> str:
    sec = int(max(sec, 0))
    h = sec // 3600
    m = (sec % 3600) // 60
    s = sec % 60
    if h > 0:
        return f'{h:d}:{m:02d}:{s:02d}'
    return f'{m:d}:{s:02d}'


def log_progress(prefix: str, idx: int, total: int, start_time: float, correct: int):
    elapsed = time.time() - start_time
    avg = elapsed / max(idx, 1)
    eta = avg * max(total - idx, 0)
    logger.info(
        '%s | %d/%d | acc=%.4f | elapsed=%s | eta=%s | %.2fs/item',
        prefix,
        idx,
        total,
        correct / max(idx, 1),
        format_seconds(elapsed),
        format_seconds(eta),
        avg,
    )


WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.


WARN  Feature `utils/Perplexity` requires Python < 3.14 and Python GIL enabled and Python >= 3.13.3T (T for Threading-Free edition of Python) plus Torch 2.8. Feature is currently skipped/disabled.


INFO  ENV: Auto setting PYTORCH_ALLOC_CONF='expandable_segments:True,max_split_size_mb:256,garbage_collection_threshold:0.7' for memory saving.


INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.          


INFO  

_____/\\\\\\\\\\\\__/\\\\\\\\\\\\\____/\\\\\\\\\\\\\\\______________________/\\\________/\\\\____________/\\\\_______________________/\\\__________________/\\\\\\____
 ___/\\\//////////__\/\\\/////////\\\_\///////\\\/////____________________/\\\\/\\\\____\/\\\\\\________/\\\\\\______________________\/\\\_________________\////\\\____
  __/\\\_____________\/\\\_______\/\\\_______\/\\\_______________________/\\\//\////\\\__\/\\\//\\\____/\\\//\\\______________________\/\\\____________________\/\\\____
   _\/\\\____/\\\\\\\_\/\\\\\\\\\\\\\/________\/\\\________/\\\\\\\\\\\__/\\\______\//\\\_\/\\\\///\\\/\\\/_\/\\\_____/\\\\\___________\/\\\______/\\\\\\\\_____\/\\\____
    _\/\\\___\/////\\\_\/\\\/////////__________\/\\\_______\///////////__\//\\\______/\\\__\/\\\__\///\\\/___\/\\\___/\\\///\\\____/\\\\\\\\\____/\\\/////\\\____\/\\\____
     _\/\\\_______\/\\\_\/\\\___________________\/\\\______________________\///\\\\/\\\\/___\/\\\____\///_____\/\\\__/\\\__\//\\\__/\\\////\\\___/\

Working directory: /content
Artifact dir: /content/artifacts/gptq_w4_g128/attn_only_quant
Results dir: /content/results/gptq_w4_g128_attn_only_quant/attn_only_quant
Bundle dir: /content/zip_bundles/gptq_w4_g128_attn_only_quant/attn_only_quant
CUDA available: True
GPU: NVIDIA L4


In [ ]:
# Quantization helpers

def get_calibration_data(cfg, tokenizer=None):
    cal_cfg = cfg['calibration']
    cache_path = CALIB_DIR / f"calibration_{cal_cfg['num_samples']}_{cal_cfg['max_length']}.pt"
    if cache_path.exists():
        logger.info('Using cached calibration samples: %s', cache_path)
        return torch.load(cache_path, weights_only=False)

    if tokenizer is None:
        tokenizer = AutoTokenizer.from_pretrained(cfg['base_model'], trust_remote_code=True, use_fast=True)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

    logger.info('Loading calibration dataset: %s / %s / %s', cal_cfg['dataset'], cal_cfg['dataset_name'], cal_cfg['split'])
    ds = load_dataset(cal_cfg['dataset'], cal_cfg['dataset_name'], split=cal_cfg['split'])
    all_text = '\n\n'.join(t for t in ds['text'] if t and t.strip())
    encoded = tokenizer(all_text, return_tensors='pt')
    total_tokens = encoded.input_ids.shape[1]
    max_len = cal_cfg['max_length']

    if total_tokens <= max_len:
        raise ValueError(f'Calibration corpus too short: total_tokens={total_tokens}, max_length={max_len}')

    rng = torch.Generator().manual_seed(cal_cfg['seed'])
    starts = torch.randint(0, total_tokens - max_len, (cal_cfg['num_samples'],), generator=rng)
    samples = []
    for start in tqdm(starts.tolist(), desc='Preparing calibration windows', unit='sample'):
        end = start + max_len
        input_ids = encoded.input_ids[:, start:end]
        samples.append({
            'input_ids': input_ids,
            'attention_mask': torch.ones_like(input_ids),
        })

    torch.save(samples, cache_path)
    logger.info('Saved calibration cache: %s', cache_path)
    return samples


def quantize_variant_gptq(cfg):
    tokenizer = AutoTokenizer.from_pretrained(cfg['base_model'], trust_remote_code=True, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    quantized_already = (ARTIFACT_DIR / 'config.json').exists() and find_gptq_model_basename(ARTIFACT_DIR) is not None
    if quantized_already:
        logger.info('Quantized artifact already exists at %s', ARTIFACT_DIR)
        zip_path = BUNDLE_DIR / f'{VARIANT}_quantized_model.zip'
        if not zip_path.exists():
            zip_directory(ARTIFACT_DIR, zip_path)
        return {
            'artifact_dir': str(ARTIFACT_DIR),
            'zip_path': str(zip_path),
            'gptq_model_file': find_gptq_model_basename(ARTIFACT_DIR),
            'dynamic_config': build_dynamic_config(VARIANT),
            'model_size_gb': round(model_size_gb(ARTIFACT_DIR), 4),
            'bits_per_param': round(bits_per_param(ARTIFACT_DIR), 4),
        }

    samples = get_calibration_data(cfg, tokenizer)
    cal_dataset = [
        {
            'input_ids': s['input_ids'].squeeze(0),
            'attention_mask': s['attention_mask'].squeeze(0),
        }
        for s in samples
    ]

    qcfg_kwargs = {
        'bits': cfg['quant']['bits'],
        'group_size': cfg['quant']['group_size'],
        'desc_act': cfg['quant'].get('desc_act', False),
    }
    dynamic = build_dynamic_config(VARIANT)
    if dynamic is not None:
        qcfg_kwargs['dynamic'] = dynamic

    quantize_config = QuantizeConfig(**qcfg_kwargs)

    logger.info('Starting GPTQ %s for %s', VARIANT, cfg['base_model'])
    t0 = time.time()
    model = GPTQModel.load(cfg['base_model'], quantize_config, trust_remote_code=True)
    model.quantize(cal_dataset)
    model.save(str(ARTIFACT_DIR))
    tokenizer.save_pretrained(str(ARTIFACT_DIR))
    elapsed = time.time() - t0
    logger.info('Quantization finished in %s', format_seconds(elapsed))

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    model_file = find_gptq_model_basename(ARTIFACT_DIR)
    if model_file is None:
        raise FileNotFoundError(f'No GPTQ safetensors found in {ARTIFACT_DIR}')

    manifest = {
        'variant': VARIANT,
        'base_model': cfg['base_model'],
        'quant': cfg['quant'],
        'dynamic_config': dynamic,
        'artifact_dir': str(ARTIFACT_DIR),
        'gptq_model_file': model_file,
        'quantization_elapsed_sec': elapsed,
        'model_size_gb': round(model_size_gb(ARTIFACT_DIR), 4),
        'bits_per_param': round(bits_per_param(ARTIFACT_DIR), 4),
        'created_at_unix': time.time(),
    }
    save_json(manifest, ARTIFACT_DIR / 'artifact_manifest.json')
    save_json(manifest, RESULTS_DIR / 'quantization_manifest.json')

    zip_path = zip_directory(ARTIFACT_DIR, BUNDLE_DIR / f'{VARIANT}_quantized_model.zip')

    state = get_results_state()
    state['quantization'] = manifest | {'zip_path': str(zip_path)}
    save_results_state(state)
    summary = update_summary_table(state)

    print('Quantized artifact ready.')
    print(json.dumps(manifest, indent=2))
    display(pd.DataFrame([summary]))
    return manifest | {'zip_path': str(zip_path)}


In [ ]:
# Cell 1 — Quantize the model and save the quantized artifact zip
quant_manifest = quantize_variant_gptq(cfg)


2026-03-09 23:55:29,702 | INFO    | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2-1.5B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-09 23:55:29,708 | INFO    | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2-1.5B/8a16abf2848eda07cc5253dec660bf1ce007ad7a/config.json "HTTP/1.1 200 OK"
2026-03-09 23:55:29,715 | INFO    | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2-1.5B/8a16abf2848eda07cc5253dec660bf1ce007ad7a/config.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

2026-03-09 23:55:30,028 | INFO    | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2-1.5B/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-03-09 23:55:30,034 | INFO    | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2-1.5B/8a16abf2848eda07cc5253dec660bf1ce007ad7a/tokenizer_config.json "HTTP/1.1 200 OK"
2026-03-09 23:55:30,041 | INFO    | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2-1.5B/8a16abf2848eda07cc5253dec660bf1ce007ad7a/tokenizer_config.json "HTTP/1.1 200 OK"


tokenizer_config.json: 0.00B [00:00, ?B/s]

2026-03-09 23:55:30,289 | INFO    | HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2-1.5B/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-03-09 23:55:30,775 | INFO    | HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2-1.5B/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-03-09 23:55:31,008 | INFO    | HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2-1.5B/resolve/main/vocab.json "HTTP/1.1 307 Temporary Redirect"
2026-03-09 23:55:31,013 | INFO    | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2-1.5B/8a16abf2848eda07cc5253dec660bf1ce007ad7a/vocab.json "HTTP/1.1 200 OK"
2026-03-09 23:55:31,020 | INFO    | HTTP Request: GET https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2-1.5B/8a16abf2848eda07cc5253dec660bf1ce007ad7a/vocab.json "HTTP/1.1 200 OK"


...

INFO  | Process finalize   | 2     | 0.000  | 0.000 | 0.001   | 0.0%   | tp-pre-pad                                           |


INFO  +--------------------+-------+--------+-------+---------+--------+------------------------------------------------------+


2026-03-10 00:00:40,097 | INFO    | Quantization finished in 4:44
2026-03-10 00:05:43,840 | INFO    | Created zip: /content/zip_bundles/gptq_w4_g128_attn_only_quant/attn_only_quant/attn_only_quant_quantized_model.zip (2172.80 MB)


Quantized artifact ready.
{
  "variant": "attn_only_quant",
  "base_model": "Qwen/Qwen2-1.5B",
  "quant": {
    "bits": 4,
    "group_size": 128,
    "desc_act": false
  },
  "dynamic_config": {
    "-:.*mlp\\.(gate|up|down)_proj.*": {}
  },
  "artifact_dir": "/content/artifacts/gptq_w4_g128/attn_only_quant",
  "gptq_model_file": "model.safetensors",
  "quantization_elapsed_sec": 284.4835362434387,
  "model_size_gb": 2.6636,
  "bits_per_param": 13.8359,
  "created_at_unix": 1773100840.1789382
}


,variant,compression,model_size_gb,peak_vram_gb,bits_per_param,perplexity,gsm8k_accuracy,arc_accuracy
0,attn_only_quant,GPTQ W4 g128,2.6636,None,13.8359,None,None,None


In [ ]:
from __future__ import annotations

import json
import logging
import math
import os
import re
import time
import zipfile
from pathlib import Path
from typing import Any

import pandas as pd
import torch
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from gptqmodel import GPTQModel, QuantizeConfig

# ------------------------------
# Logging
# ------------------------------
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-7s | %(message)s',
    force=True,
)
logger = logging.getLogger('gptq_attn_only_quant')

PROJECT_ROOT = Path.cwd().resolve()
VARIANT = 'attn_only_quant'
RUN_NAME = 'gptq_w4_g128_attn_only_quant'
BASE_MODEL = 'Qwen/Qwen2-1.5B'

cfg = {
    'base_model': BASE_MODEL,
    'calibration': {
        'dataset': 'wikitext',
        'dataset_name': 'wikitext-2-raw-v1',
        'split': 'train',
        'num_samples': 512,
        'max_length': 2048,
        'seed': 42,
    },
    'quant': {
        'bits': 4,
        'group_size': 128,
        'desc_act': False,
    },
    'eval': {
        'perplexity': {
            'dataset': 'wikitext',
            'dataset_name': 'wikitext-2-raw-v1',
            'split': 'test',
            'max_length': 2048,
            'stride': 1024,
            'max_eval_tokens': 131072,
            'log_every': 25,
        },
        'gsm8k': {
            'num_fewshot': 8,
            'num_samples': 300,
            'max_new_tokens': 256,
            'log_every': 25,
        },
        'arc_challenge': {
            'num_samples': 500,
            'max_new_tokens': 5,
            'log_every': 50,
        },
    },
    'paths': {
        'artifacts_dir': str(PROJECT_ROOT / 'artifacts'),
        'results_dir': str(PROJECT_ROOT / 'results' / RUN_NAME),
        'calibration_dir': str(PROJECT_ROOT / 'calibration_data'),
        'bundles_dir': str(PROJECT_ROOT / 'zip_bundles' / RUN_NAME),
    },
}

PATHS = {k: Path(v) for k, v in cfg['paths'].items()}
for p in PATHS.values():
    p.mkdir(parents=True, exist_ok=True)

ARTIFACT_DIR = PATHS['artifacts_dir'] / 'gptq_w4_g128' / VARIANT
RESULTS_DIR = PATHS['results_dir'] / VARIANT
BUNDLE_DIR = PATHS['bundles_dir'] / VARIANT
CALIB_DIR = PATHS['calibration_dir']
for p in [ARTIFACT_DIR, RESULTS_DIR, BUNDLE_DIR, CALIB_DIR]:
    p.mkdir(parents=True, exist_ok=True)


def build_dynamic_config(variant: str) -> dict | None:
    if variant == 'full_quant':
        return None
    if variant == 'attn_only_quant':
        return {r'-:.*mlp\.(gate|up|down)_proj.*': {}}
    if variant == 'mlp_only_quant':
        return {r'-:.*self_attn\.(q|k|v|o)_proj.*': {}}
    raise ValueError(f'Unknown variant: {variant}')


print('Working directory:', PROJECT_ROOT)
print('Artifact dir:', ARTIFACT_DIR)
print('Results dir:', RESULTS_DIR)
print('Bundle dir:', BUNDLE_DIR)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

# ------------------------------
# Small utilities
# ------------------------------
def save_json(obj: Any, path: str | Path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w') as f:
        json.dump(obj, f, indent=2)
    return path


def read_json(path: str | Path, default=None):
    path = Path(path)
    if not path.exists():
        return default
    with open(path) as f:
        return json.load(f)


def zip_directory(source_dir: str | Path, zip_path: str | Path, *, exclude_suffixes=('.lock',), exclude_dirs=('.ipynb_checkpoints',)):
    source_dir = Path(source_dir).resolve()
    zip_path = Path(zip_path).resolve()
    zip_path.parent.mkdir(parents=True, exist_ok=True)

    tmp_zip = zip_path.with_suffix(zip_path.suffix + '.tmp')
    if tmp_zip.exists():
        tmp_zip.unlink()
    if zip_path.exists():
        zip_path.unlink()

    with zipfile.ZipFile(tmp_zip, 'w', compression=zipfile.ZIP_DEFLATED, allowZip64=True) as zf:
        for path in sorted(source_dir.rglob('*')):
            if path.is_dir():
                continue
            if any(part in exclude_dirs for part in path.parts):
                continue
            if path.resolve() in {zip_path, tmp_zip}:
                continue
            if path.suffix in exclude_suffixes:
                continue
            zf.write(path, arcname=str(path.relative_to(source_dir)))

    tmp_zip.replace(zip_path)
    size_mb = zip_path.stat().st_size / (1024 ** 2)
    logger.info('Created zip: %s (%.2f MB)', zip_path, size_mb)
    return zip_path


def find_gptq_model_basename(model_dir: str | Path) -> str | None:
    model_dir = Path(model_dir)
    candidates = sorted(model_dir.glob('*.safetensors'))
    if not candidates:
        return None
    preferred = [p for p in candidates if 'gptq' in p.name.lower()]
    return (preferred[0] if preferred else candidates[0]).name


def estimate_num_params_from_config(model_dir: str | Path) -> int | None:
    cfg_path = Path(model_dir) / 'config.json'
    if not cfg_path.exists():
        return None
    cfg_json = json.loads(cfg_path.read_text())
    hidden_size = cfg_json.get('hidden_size')
    num_hidden_layers = cfg_json.get('num_hidden_layers')
    intermediate_size = cfg_json.get('intermediate_size')
    vocab_size = cfg_json.get('vocab_size')
    if not all(v is not None for v in [hidden_size, num_hidden_layers, intermediate_size, vocab_size]):
        return None
    rough_params = (
        vocab_size * hidden_size +
        num_hidden_layers * (4 * hidden_size * hidden_size) +
        num_hidden_layers * (3 * hidden_size * intermediate_size)
    )
    return int(rough_params)


def model_size_gb(model_dir: str | Path) -> float:
    files = list(Path(model_dir).glob('*.safetensors'))
    total_bytes = sum(p.stat().st_size for p in files)
    return total_bytes / (1024 ** 3)


def bits_per_param(model_dir: str | Path) -> float | None:
    files = list(Path(model_dir).glob('*.safetensors'))
    if not files:
        return None
    total_bytes = sum(p.stat().st_size for p in files)
    n_params = estimate_num_params_from_config(model_dir)
    if not n_params:
        return None
    return 8.0 * total_bytes / n_params


def get_results_state() -> dict:
    state = read_json(RESULTS_DIR / 'state.json', default={})
    return state or {}


def save_results_state(state: dict):
    save_json(state, RESULTS_DIR / 'state.json')
    return state


def update_summary_table(state: dict):
    size_gb = model_size_gb(ARTIFACT_DIR) if ARTIFACT_DIR.exists() else None
    bpp = bits_per_param(ARTIFACT_DIR) if ARTIFACT_DIR.exists() else None
    peak_candidates = []
    for key in ['perplexity', 'gsm8k', 'arc_challenge']:
        if key in state and isinstance(state[key], dict):
            val = state[key].get('peak_vram_gb')
            if val is not None:
                peak_candidates.append(val)

    summary_row = {
        'variant': VARIANT,
        'compression': f"GPTQ W{cfg['quant']['bits']} g{cfg['quant']['group_size']}",
        'model_size_gb': round(size_gb, 4) if size_gb is not None else None,
        'peak_vram_gb': round(max(peak_candidates), 4) if peak_candidates else None,
        'bits_per_param': round(bpp, 4) if bpp is not None else None,
        'perplexity': state.get('perplexity', {}).get('perplexity'),
        'gsm8k_accuracy': state.get('gsm8k', {}).get('accuracy'),
        'arc_accuracy': state.get('arc_challenge', {}).get('accuracy'),
    }
    save_json(summary_row, RESULTS_DIR / 'summary.json')
    pd.DataFrame([summary_row]).to_csv(RESULTS_DIR / 'summary.csv', index=False)
    return summary_row


def _is_gptq_checkpoint(model_path: str | Path) -> bool:
    cfg_path = Path(model_path) / 'config.json'
    if not cfg_path.exists():
        return False
    cfg_json = json.loads(cfg_path.read_text())
    return cfg_json.get('quantization_config', {}).get('quant_method') == 'gptq'


def load_quantized_model_and_tokenizer(model_path: str | Path):
    model_path = Path(model_path)
    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = 'left'

    if not _is_gptq_checkpoint(model_path):
        raise ValueError(f'{model_path} is not a GPTQ checkpoint.')

    model_basename = find_gptq_model_basename(model_path)
    if model_basename is None:
        raise FileNotFoundError(f'No GPTQ safetensors found in {model_path}')

    model = GPTQModel.load(
        str(model_path),
        device_map='auto',
        trust_remote_code=True,
        model_basename=model_basename,
    )
    model.eval()
    return model, tokenizer


def get_model_device(model):
    if hasattr(model, 'device'):
        return model.device
    if hasattr(model, 'hf_device_map') and model.hf_device_map:
        first_device = next(iter(model.hf_device_map.values()))
        return torch.device(first_device)
    return torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')


def generate_text(model, tokenizer, prompt: str, *, max_new_tokens: int):
    device = get_model_device(model)
    encoded = tokenizer(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=min(getattr(tokenizer, 'model_max_length', 4096), 4096),
        padding=False,
    )
    input_ids = encoded['input_ids'].to(device)
    attention_mask = encoded['attention_mask'].to(device)

    with torch.inference_mode():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )
    gen_ids = outputs[0][input_ids.shape[-1]:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True)


ANSWER_RE = re.compile(r'####\s*(-?[\d,]+\.?\d*)')


def extract_numeric_answer(text: str):
    m = ANSWER_RE.search(text)
    if m:
        try:
            return float(m.group(1).replace(',', ''))
        except ValueError:
            return None
    nums = re.findall(r'-?[\d,]+\.?\d*', text)
    if not nums:
        return None
    try:
        return float(nums[-1].replace(',', ''))
    except ValueError:
        return None


def format_seconds(sec: float) -> str:
    sec = int(max(sec, 0))
    h = sec // 3600
    m = (sec % 3600) // 60
    s = sec % 60
    if h > 0:
        return f'{h:d}:{m:02d}:{s:02d}'
    return f'{m:d}:{s:02d}'


def log_progress(prefix: str, idx: int, total: int, start_time: float, correct: int):
    elapsed = time.time() - start_time
    avg = elapsed / max(idx, 1)
    eta = avg * max(total - idx, 0)
    logger.info(
        '%s | %d/%d | acc=%.4f | elapsed=%s | eta=%s | %.2fs/item',
        prefix,
        idx,
        total,
        correct / max(idx, 1),
        format_seconds(elapsed),
        format_seconds(eta),
        avg,
    )


GSM_PROMPT = """Question: {question}
Answer: Let's think step by step.
"""

def evaluate_gsm8k_only(model, tokenizer, cfg):
    gsm_cfg = cfg['eval']['gsm8k']
    n_shot = gsm_cfg['num_fewshot']
    num_samples = gsm_cfg['num_samples']
    max_new = gsm_cfg['max_new_tokens']
    log_every = gsm_cfg.get('log_every', 25)

    logger.info('Loading GSM8K: %d-shot, %d samples, max_new_tokens=%d', n_shot, num_samples, max_new)
    train_ds = load_dataset('openai/gsm8k', 'main', split='train').shuffle(seed=42)
    test_ds = load_dataset('openai/gsm8k', 'main', split='test').shuffle(seed=42).select(range(num_samples))
    exemplars = train_ds.select(range(n_shot))

    prefix = ''
    for ex in exemplars:
        prefix += GSM_PROMPT.format(question=ex['question']) + ex['answer'] + '\n\n'

    rows = []
    correct = 0
    start_time = time.time()
    peak_vram = 0.0
    total = len(test_ds)

    for idx, row in enumerate(tqdm(test_ds, desc='GSM8K', unit='q'), start=1):
        prompt = prefix + GSM_PROMPT.format(question=row['question'])

        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()

        generated = generate_text(model, tokenizer, prompt, max_new_tokens=max_new)
        pred = extract_numeric_answer(generated)
        gold = extract_numeric_answer(row['answer'])
        is_correct = pred is not None and gold is not None and abs(pred - gold) < 1e-3
        correct += int(is_correct)

        if torch.cuda.is_available():
            peak_vram = max(peak_vram, torch.cuda.max_memory_allocated() / (1024 ** 3))

        rows.append({
            'index': idx - 1,
            'question': row['question'],
            'prediction_text': generated,
            'predicted_value': pred,
            'gold_value': gold,
            'correct': bool(is_correct),
        })

        if idx % log_every == 0 or idx == total:
            log_progress('GSM8K', idx, total, start_time, correct)

    elapsed = time.time() - start_time
    result = {
        'dataset': 'gsm8k',
        'num_fewshot': n_shot,
        'num_samples': total,
        'max_new_tokens': max_new,
        'accuracy': correct / max(total, 1),
        'correct': correct,
        'elapsed_sec': elapsed,
        'elapsed_hms': format_seconds(elapsed),
        'peak_vram_gb': peak_vram if torch.cuda.is_available() else None,
    }

    pd.DataFrame(rows).to_csv(RESULTS_DIR / 'gsm8k_predictions.csv', index=False)
    save_json(result, RESULTS_DIR / 'gsm8k_metrics.json')

    state = get_results_state()
    state['gsm8k'] = result
    save_results_state(state)
    summary = update_summary_table(state)

    logger.info('GSM8K finished | acc=%.4f | elapsed=%s | peak_vram_gb=%s', result['accuracy'], result['elapsed_hms'], result['peak_vram_gb'])
    display(pd.DataFrame([result]))
    display(pd.DataFrame([summary]))
    return result


def format_arc_prompt(row):
    labels = row['choices']['label']
    texts = row['choices']['text']
    prompt = row['question'] + '\n'
    for lbl, txt in zip(labels, texts):
        prompt += f'{lbl}. {txt}\n'
    prompt += 'Answer:'
    return prompt


def evaluate_arc_only(model, tokenizer, cfg):
    arc_cfg = cfg['eval']['arc_challenge']
    num_samples = arc_cfg['num_samples']
    max_new = arc_cfg['max_new_tokens']
    log_every = arc_cfg.get('log_every', 50)

    logger.info('Loading ARC-Challenge: %d samples, max_new_tokens=%d', num_samples, max_new)
    ds = load_dataset('allenai/ai2_arc', 'ARC-Challenge', split='test').shuffle(seed=42).select(range(num_samples))

    rows = []
    correct = 0
    start_time = time.time()
    peak_vram = 0.0
    total = len(ds)

    for idx, row in enumerate(tqdm(ds, desc='ARC-Challenge', unit='q'), start=1):
        prompt = format_arc_prompt(row)

        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()

        generated = generate_text(model, tokenizer, prompt, max_new_tokens=max_new).strip()
        pred = generated[:1].upper() if generated else None
        gold = row['answerKey'].strip().upper()
        is_correct = pred == gold
        correct += int(is_correct)

        if torch.cuda.is_available():
            peak_vram = max(peak_vram, torch.cuda.max_memory_allocated() / (1024 ** 3))

        rows.append({
            'index': idx - 1,
            'question': row['question'],
            'prediction_text': generated,
            'predicted_label': pred,
            'gold_label': gold,
            'correct': bool(is_correct),
        })

        if idx % log_every == 0 or idx == total:
            log_progress('ARC-Challenge', idx, total, start_time, correct)

    elapsed = time.time() - start_time
    result = {
        'dataset': 'arc_challenge',
        'num_samples': total,
        'max_new_tokens': max_new,
        'accuracy': correct / max(total, 1),
        'correct': correct,
        'elapsed_sec': elapsed,
        'elapsed_hms': format_seconds(elapsed),
        'peak_vram_gb': peak_vram if torch.cuda.is_available() else None,
    }

    pd.DataFrame(rows).to_csv(RESULTS_DIR / 'arc_challenge_predictions.csv', index=False)
    save_json(result, RESULTS_DIR / 'arc_challenge_metrics.json')

    state = get_results_state()
    state['arc_challenge'] = result
    save_results_state(state)
    summary = update_summary_table(state)

    results_zip = zip_directory(RESULTS_DIR, BUNDLE_DIR / 'full_quant_results_only.zip')
    state['results_zip'] = str(results_zip)
    save_results_state(state)

    logger.info('ARC-Challenge finished | acc=%.4f | elapsed=%s | peak_vram_gb=%s', result['accuracy'], result['elapsed_hms'], result['peak_vram_gb'])
    display(pd.DataFrame([result]))
    display(pd.DataFrame([summary]))
    print('Results zip:', results_zip)

Working directory: /content
Artifact dir: /content/artifacts/gptq_w4_g128/attn_only_quant
Results dir: /content/results/gptq_w4_g128_attn_only_quant/attn_only_quant
Bundle dir: /content/zip_bundles/gptq_w4_g128_attn_only_quant/attn_only_quant
CUDA available: True
GPU: NVIDIA L4


In [ ]:
import os
import json
import time
import math
import torch
from datasets import load_dataset


def evaluate_perplexity_only(model, tokenizer, results_dir):

    print("=" * 80)
    print("PERPLEXITY EVALUATION")
    print("=" * 80)

    dataset_name = "wikitext"
    dataset_config = "wikitext-2-raw-v1"
    split = "test"

    max_length = 2048
    stride = 1024

    print(f"Dataset: {dataset_name}/{dataset_config}")
    print(f"Split: {split}")
    print(f"Max length: {max_length}")
    print(f"Stride: {stride}")

    os.makedirs(results_dir, exist_ok=True)

    print("\nLoading dataset...")
    dataset = load_dataset(dataset_name, dataset_config, split=split)

    text = "\n\n".join(dataset["text"])

    print("Tokenizing...")
    encodings = tokenizer(text, return_tensors="pt")

    input_ids = encodings.input_ids
    seq_len = input_ids.size(1)

    print(f"Total tokens: {seq_len}")

    nlls = []
    prev_end_loc = 0

    start_time = time.time()

    total_steps = math.ceil((seq_len - max_length) / stride) + 1

    step = 0

    for begin_loc in range(0, seq_len, stride):

        end_loc = min(begin_loc + max_length, seq_len)
        trg_len = end_loc - prev_end_loc

        input_ids_slice = input_ids[:, begin_loc:end_loc].to(model.device)

        target_ids = input_ids_slice.clone()
        target_ids[:, :-trg_len] = -100

        with torch.no_grad():

            outputs = model(
                input_ids=input_ids_slice,
                labels=target_ids
            )

            neg_log_likelihood = outputs.loss

        nlls.append(neg_log_likelihood * trg_len)

        prev_end_loc = end_loc

        step += 1

        elapsed = time.time() - start_time
        avg = elapsed / step
        eta = avg * (total_steps - step)

        print(
            f"[PPL] step {step}/{total_steps} | "
            f"tokens {begin_loc}-{end_loc} | "
            f"elapsed {elapsed/60:.2f} min | "
            f"eta {eta/60:.2f} min"
        )

        if end_loc == seq_len:
            break

    ppl = torch.exp(torch.stack(nlls).sum() / end_loc).item()

    metrics = {
        "dataset": "wikitext-2",
        "split": "test",
        "max_length": max_length,
        "stride": stride,
        "perplexity": float(ppl)
    }

    with open(os.path.join(results_dir, "perplexity_metrics.json"), "w") as f:
        json.dump(metrics, f, indent=2)

    print("\nPerplexity:", ppl)

    return metrics

In [ ]:
# Cell 2 — Evaluate Perplexity only
# Run this after the quantization cell.
if not ARTIFACT_DIR.exists():
    raise FileNotFoundError(f'Quantized artifact not found: {ARTIFACT_DIR}')

model, tokenizer = load_quantized_model_and_tokenizer(ARTIFACT_DIR)
try:
    perplexity_result = evaluate_perplexity_only(model, tokenizer, RESULTS_DIR)
finally:
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

from_quantized: adapter: None


INFO  Loader: Auto dtype (native bfloat16): `torch.bfloat16`                   


INFO  QuantizeConfig: offload_to_disk_path auto set to `./gptqmodel_offload/ywcpdvot-tkcxvfua/`


INFO  Estimated Quantization BPW (bits per weight): 4.2875 bpw, based on [bits: 4, group_size: 128]


INFO  The layer model.layers.0.mlp.gate_proj is not quantized.                 


...

PERPLEXITY EVALUATION
Dataset: wikitext/wikitext-2-raw-v1
Split: test
Max length: 2048
Stride: 1024

Loading dataset...


2026-03-10 00:15:48,335 | INFO    | HTTP Request: HEAD https://huggingface.co/datasets/wikitext/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-03-10 00:15:48,566 | INFO    | HTTP Request: HEAD https://huggingface.co/datasets/Salesforce/wikitext/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-03-10 00:15:48,572 | INFO    | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/Salesforce/wikitext/b08601e04326c79dfdd32d625aee71d232d685c3/README.md "HTTP/1.1 200 OK"
2026-03-10 00:15:48,873 | INFO    | HTTP Request: HEAD https://huggingface.co/datasets/wikitext/resolve/b08601e04326c79dfdd32d625aee71d232d685c3/wikitext.py "HTTP/1.1 307 Temporary Redirect"
2026-03-10 00:15:49,114 | INFO    | HTTP Request: HEAD https://huggingface.co/datasets/Salesforce/wikitext/resolve/b08601e04326c79dfdd32d625aee71d232d685c3/wikitext.py "HTTP/1.1 404 Not Found"
2026-03-10 00:15:49,806 | INFO    | HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/d

Tokenizing...


Token indices sequence length is longer than the specified maximum sequence length for this model (299078 > 32768). Running this sequence through the model will result in indexing errors


Total tokens: 299078
[PPL] step 1/292 | tokens 0-2048 | elapsed 0.00 min | eta 0.92 min
[PPL] step 2/292 | tokens 1024-3072 | elapsed 0.00 min | eta 0.63 min
[PPL] step 3/292 | tokens 2048-4096 | elapsed 0.01 min | eta 0.70 min
[PPL] step 4/292 | tokens 3072-5120 | elapsed 0.01 min | eta 0.74 min
[PPL] step 5/292 | tokens 4096-6144 | elapsed 0.01 min | eta 0.77 min
[PPL] step 6/292 | tokens 5120-7168 | elapsed 0.02 min | eta 0.79 min
[PPL] step 7/292 | tokens 6144-8192 | elapsed 0.02 min | eta 0.80 min
[PPL] step 8/292 | tokens 7168-9216 | elapsed 0.02 min | eta 0.81 min
[PPL] step 9/292 | tokens 8192-10240 | elapsed 0.03 min | eta 0.81 min
[PPL] step 10/292 | tokens 9216-11264 | elapsed 0.03 min | eta 0.82 min
[PPL] step 11/292 | tokens 10240-12288 | elapsed 0.03 min | eta 0.82 min
[PPL] step 12/292 | tokens 11264-13312 | elapsed 0.04 min | eta 0.82 min
[PPL] step 13/292 | tokens 12288-14336 | elapsed 0.04 min | eta 0.82 min
[PPL] step 14/292 | tokens 13312-15360 | elapsed 0.04 min | 

In [ ]:

# Cell 3 — Evaluate on GSM8K only
# Run this after the Perplexity cell.
if not ARTIFACT_DIR.exists():
    raise FileNotFoundError(f'Quantized artifact not found: {ARTIFACT_DIR}')

model, tokenizer = load_quantized_model_and_tokenizer(ARTIFACT_DIR)
try:
    gsm8k_result = evaluate_gsm8k_only(model, tokenizer, cfg)
finally:
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [ ]:

# Cell 4 — Evaluate on ARC-Challenge only and finalize the summary/results zip
# Run this after the GSM8K cell.
if not ARTIFACT_DIR.exists():
    raise FileNotFoundError(f'Quantized artifact not found: {ARTIFACT_DIR}')

model, tokenizer = load_quantized_model_and_tokenizer(ARTIFACT_DIR)
try:
    arc_result = evaluate_arc_only(model, tokenizer, cfg)
finally:
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


from_quantized: adapter: None


INFO  Loader: Auto dtype (native bfloat16): `torch.bfloat16`                   


INFO  QuantizeConfig: offload_to_disk_path auto set to `./gptqmodel_offload/pnnvijib-iqbxrhpb/`


INFO  Estimated Quantization BPW (bits per weight): 4.2875 bpw, based on [bits: 4, group_size: 128]


INFO  The layer model.layers.0.mlp.gate_proj is not quantized.                 


...

ARC-Challenge:   0%|          | 0/500 [00:00<?, ?q/s]

2026-03-10 00:13:09,851 | INFO    | ARC-Challenge | 50/500 | acc=0.6400 | elapsed=0:17 | eta=2:39 | 0.35s/item
2026-03-10 00:13:24,615 | INFO    | ARC-Challenge | 100/500 | acc=0.6800 | elapsed=0:32 | eta=2:09 | 0.32s/item
2026-03-10 00:13:39,345 | INFO    | ARC-Challenge | 150/500 | acc=0.6867 | elapsed=0:47 | eta=1:50 | 0.31s/item
2026-03-10 00:13:54,084 | INFO    | ARC-Challenge | 200/500 | acc=0.6550 | elapsed=1:01 | eta=1:32 | 0.31s/item
2026-03-10 00:14:08,770 | INFO    | ARC-Challenge | 250/500 | acc=0.6520 | elapsed=1:16 | eta=1:16 | 0.31s/item
2026-03-10 00:14:23,492 | INFO    | ARC-Challenge | 300/500 | acc=0.6433 | elapsed=1:31 | eta=1:00 | 0.30s/item
2026-03-10 00:14:38,144 | INFO    | ARC-Challenge | 350/500 | acc=0.6314 | elapsed=1:46 | eta=0:45 | 0.30s/item
2026-03-10 00:14:52,784 | INFO    | ARC-Challenge | 400/500 | acc=0.6375 | elapsed=2:00 | eta=0:30 | 0.30s/item
2026-03-10 00:15:07,405 | INFO    | ARC-Challenge | 450/500 | acc=0.6444 | elapsed=2:15 | eta=0:15 | 0.30

,dataset,num_samples,max_new_tokens,accuracy,correct,elapsed_sec,elapsed_hms,peak_vram_gb
0,arc_challenge,500,5,0.648,324,149.871307,2:29,11.1823


,variant,compression,model_size_gb,peak_vram_gb,bits_per_param,perplexity,gsm8k_accuracy,arc_accuracy
0,attn_only_quant,GPTQ W4 g128,2.6636,11.1823,13.8359,None,None,0.648


Results zip: /content/zip_bundles/gptq_w4_g128_attn_only_quant/attn_only_quant/full_quant_results_only.zip


In [16]:
# Optional: inspect the final table row (Perplexity + GSM8K + ARC) and all saved files
state = get_results_state()
summary = update_summary_table(state)
print(json.dumps(state, indent=2))
display(pd.DataFrame([summary]))
print('\nSaved files under results:')
for p in sorted(RESULTS_DIR.glob('*')):
    print('-', p.name)
print('\nSaved bundles:')
for p in sorted(BUNDLE_DIR.glob('*')):
    print('-', p.name)

{
  "quantization": {
    "variant": "attn_only_quant",
    "base_model": "Qwen/Qwen2-1.5B",
    "quant": {
      "bits": 4,
      "group_size": 128,
      "desc_act": false
    },
    "dynamic_config": {
      "-:.*mlp\\.(gate|up|down)_proj.*": {}
    },
    "artifact_dir": "/content/artifacts/gptq_w4_g128/attn_only_quant",
    "gptq_model_file": "model.safetensors",
    "quantization_elapsed_sec": 284.4835362434387,
    "model_size_gb": 2.6636,
    "bits_per_param": 13.8359,
    "created_at_unix": 1773100840.1789382,
    "zip_path": "/content/zip_bundles/gptq_w4_g128_attn_only_quant/attn_only_quant/attn_only_quant_quantized_model.zip"
  },
  "arc_challenge": {
    "dataset": "arc_challenge",
    "num_samples": 500,
    "max_new_tokens": 5,
    "accuracy": 0.648,
    "correct": 324,
    "elapsed_sec": 149.8713071346283,
    "elapsed_hms": "2:29",
    "peak_vram_gb": 11.182299613952637
  },
  "results_zip": "/content/zip_bundles/gptq_w4_g128_attn_only_quant/attn_only_quant/full_quant_r

,variant,compression,model_size_gb,peak_vram_gb,bits_per_param,perplexity,gsm8k_accuracy,arc_accuracy
0,attn_only_quant,GPTQ W4 g128,2.6636,11.1823,13.8359,None,None,0.648



Saved files under results:
- arc_challenge_metrics.json
- arc_challenge_predictions.csv
- perplexity_metrics.json
- quantization_manifest.json
- state.json
- summary.csv
- summary.json

Saved bundles:
- attn_only_quant_quantized_model.zip
- full_quant_results_only.zip
